# {{PROJECT_NAME}} — 01 Data Understanding

**Stage:** Data acquisition → Data understanding → Data validation

Everything in this notebook that runs automatically is *mechanical*: shape, dtypes,
missingness, duplicates, outlier flags. It is safe on any tabular dataset and tells you
nothing about what the data means.

The `## DECISION` cells are where the analysis actually starts. They are blank on purpose.

> **This notebook is a stage template, not a mandatory step.** Use the notebooks your
> question needs and delete the rest — a descriptive project may never open
> `04_modeling.ipynb`, and a messy dataset may spend most of its life in `01`.
> See `CHECKLIST.md` for the full workflow.

In [ ]:
import sys, pathlib

ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.visualization.theme import PALETTE, save_fig, set_plot_style
set_plot_style()

# Point this at your file in data/raw/. Everything below follows from it.
DATASET = 'REPLACE_ME.csv'

## DECISION 1 — What question is this data supposed to answer?

Answer before you look at a single chart. A dataset explored without a question produces
a tour, not an analysis.

- **Question:** _<what are you actually trying to find out?>_
- **Who needs the answer:** _<who reads this, and what do they do differently afterwards?>_
- **What would change your mind:** _<what result would make you abandon the hypothesis?>_

In [ ]:
from src.data.loading import list_raw, load_raw

print('files in data/raw/:', list_raw())
df = load_raw(DATASET)
df.head()

### Reading

_<What is one row? What population does it cover, and over what period? Where did it come
from and who collected it? If you cannot answer these, find out before continuing.>_

In [ ]:
from src.data.quality import profile, schema_report

profile(df)
schema_report(df)

### Reading

_<Any column typed `object` that should be numeric? Any dtype that surprises you? Units —
are they what you assumed? Note anything that needs cleaning in 02.>_

In [ ]:
from src.data.quality import missing_report
from src.visualization.plots import missingness_matrix

missing = missing_report(df)
display(missing)

if df.isna().any().any():
    fig, _ = missingness_matrix(df)
    save_fig(fig, '01_missingness')
    plt.show()
else:
    print('No missing values — skip the missingness DECISION below.')

## DECISION 2 — Why is the data missing, and does that permit imputation?

Counting gaps is mechanical; explaining them is not, and it decides whether filling them
is legitimate at all.

| Mechanism | Meaning | Imputation |
|---|---|---|
| MCAR | Missing at random, unrelated to anything | Safe; you lose precision, not validity |
| MAR | Missingness explained by *other observed* columns | Defensible if you model it on those columns |
| MNAR | Missingness depends on the missing value itself (high earners skip the income box) | Imputing here manufactures a convenient answer |

- **Mechanism, and the evidence for it:** _<blocks in the matrix above? correlated with a group?>_
- **Decision:** _<impute / drop rows / leave as NaN and report>_ because _<reason>_

Nothing downstream should fill these values until this cell is answered.

In [ ]:
from src.data.quality import cardinality_report, duplicate_report

# subset= names the columns that identify a record. Pass it once you know what a
# duplicate means here — it is a fact about your data, not something to infer.
duplicate_report(df, subset=None)
cardinality_report(df)

### Reading

_<Are the exact duplicates real repeated observations or an export artifact? Any constant
column (no information) or near-unique text column (an identifier, not a feature)?>_

In [ ]:
from src.data.quality import outlier_report

outliers = outlier_report(df)
outliers

## DECISION 3 — Are the flagged values errors or real observations?

`outlier_report` flags and never removes. A flagged value can be a typo, a unit mix-up, or
the single most interesting row in the dataset.

- **Values examined:** _<which, in which columns>_
- **Decision:** _<keep all / correct specific rows / exclude with a stated rule>_ because _<reason>_
- **Effect if you are wrong:** _<what breaks downstream>_

Excluding a row is a claim that it does not belong to the population you are studying.
Write down the claim.

In [ ]:
from src.data.quality import quality_report
from src.visualization.presentation import save_table

summary = quality_report(df)
save_table(summary, '01_data_quality')

## What this data can and cannot answer

Fill this in before moving on. It is the most useful cell in the notebook and the easiest
to skip.

**Can answer:**
- _<question, and the columns that support it>_

**Cannot answer:**
- _<question, and what is missing — no time dimension, no control group, n too small, no
  outcome variable>_

**Next stage:** _<02_eda / straight to 03_analysis / more cleaning first>_ because _<reason>_